# 03 - Transformação da Camada Silver

## Objetivo

Transformar os dados armazenados na camada Bronze em um conjunto de dados limpo, padronizado e corretamente tipado.

As regras de transformação utilizadas nesta etapa foram definidas a partir da análise de qualidade realizada anteriormente.

A camada Silver terá como principais objetivos:

- corrigir os tipos de dados;
- padronizar categorias inconsistentes;
- tratar marcadores de ausência de informação;
- criar atributos derivados úteis às análises;
- preservar a rastreabilidade dos registros;
- persistir os dados tratados em formato Delta para utilização nas etapas seguintes do pipeline.

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table(
    "workspace.default.prf_acidentes_bronze"
)

print(f"Registros na Bronze: {df_bronze.count()}")
print(f"Colunas na Bronze: {len(df_bronze.columns)}")

Registros na Bronze: 342624
Colunas na Bronze: 32


In [0]:
df_silver = df_bronze

## Conversão dos Tipos de Dados

Na camada Bronze, os dados foram mantidos em seu formato original, com os campos armazenados como texto.

Nesta etapa, os atributos serão convertidos para tipos mais adequados ao seu significado e uso analítico.

As conversões incluem:

- datas para tipo `date`;
- campos de contagem para `integer`;
- campos geográficos e quilometragem para tipos numéricos;
- preservação dos campos categóricos como texto.

As conversões foram definidas com base nos testes de consistência realizados durante a etapa de profiling.

In [0]:
df_silver = (
    df_silver

    .withColumn(
        "data_inversa",
        F.to_date(F.col("data_inversa"), "yyyy-MM-dd")
    )

    .withColumn(
        "br",
        F.col("br").cast("int")
    )

    .withColumn(
        "km",
        F.regexp_replace(F.col("km"), ",", ".").cast("double")
    )

    .withColumn(
        "pessoas",
        F.col("pessoas").cast("int")
    )

    .withColumn(
        "mortos",
        F.col("mortos").cast("int")
    )

    .withColumn(
        "feridos_leves",
        F.col("feridos_leves").cast("int")
    )

    .withColumn(
        "feridos_graves",
        F.col("feridos_graves").cast("int")
    )

    .withColumn(
        "ilesos",
        F.col("ilesos").cast("int")
    )

    .withColumn(
        "ignorados",
        F.col("ignorados").cast("int")
    )

    .withColumn(
        "feridos",
        F.col("feridos").cast("int")
    )

    .withColumn(
        "veiculos",
        F.col("veiculos").cast("int")
    )

    .withColumn(
        "latitude",
        F.regexp_replace(F.col("latitude"), ",", ".").cast("double")
    )

    .withColumn(
        "longitude",
        F.regexp_replace(F.col("longitude"), ",", ".").cast("double")
    )
)

In [0]:
df_silver.printSchema()

root
 |-- id: string (nullable = true)
 |-- data_inversa: date (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: integer (nullable = true)
 |-- mortos: integer (nullable = true)
 |-- feridos_leves: integer (nullable = true)
 |-- feridos_graves: integer (nullable = true)
 |-- ilesos: integer (nullable = true)
 |-- ignorados: integer (nullable = true)
 |-- feridos: integer (nullable = 

In [0]:
print(f"Registros antes da transformação: {df_bronze.count()}")
print(f"Registros após a transformação: {df_silver.count()}")

Registros antes da transformação: 342624
Registros após a transformação: 342624


In [0]:
colunas_convertidas = [
    "data_inversa",
    "br",
    "km",
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos",
    "latitude",
    "longitude"
]

In [0]:
display(
    df_silver.select([
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in colunas_convertidas
    ])
)

data_inversa,br,km,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude
0,0,0,0,0,0,0,0,0,0,0,0,0


### Resultado da conversão de tipos

Os campos numéricos, geográficos e temporais foram convertidos para tipos compatíveis com seu significado.

Após as conversões, a quantidade total de registros permaneceu inalterada em 342.624.

Também foi verificado se as conversões geraram novos valores nulos. Como os formatos já haviam sido validados durante a etapa de profiling, não foram observadas perdas de informação decorrentes da alteração dos tipos.

Essa etapa permite que os dados sejam utilizados corretamente em operações matemáticas, filtros temporais, agregações e análises posteriores.

## Padronização de Valores Categóricos e Ausências

Durante a etapa de profiling foram identificados valores textuais utilizados para representar ausência de informação, como `IGNORADO`, `NÃO INFORMADO`, `NA` e `N/A`.

Também foi identificada uma inconsistência de capitalização na coluna `causa_acidente`, em que a mesma categoria aparecia escrita como `Transitar no Acostamento` e `Transitar no acostamento`.

Nesta etapa, serão aplicadas regras de padronização para melhorar a consistência dos dados, preservando o significado original dos registros.

In [0]:
df_silver = df_silver.withColumn(
    "causa_acidente",
    F.when(
        F.upper(F.trim(F.col("causa_acidente"))) == "TRANSITAR NO ACOSTAMENTO",
        "Transitar no Acostamento"
    ).otherwise(F.col("causa_acidente"))
)

In [0]:
display(
    df_silver.groupBy("causa_acidente")
    .count()
    .filter(
        F.upper(F.trim(F.col("causa_acidente"))) == "TRANSITAR NO ACOSTAMENTO"
    )
)

causa_acidente,count
Transitar no Acostamento,2080


In [0]:
colunas_ausencia = [
    "condicao_metereologica",
    "sentido_via",
    "regional",
    "delegacia",
    "uop",
    "classificacao_acidente"
]

In [0]:
marcadores_ausencia = [
    "IGNORADO",
    "NÃO INFORMADO",
    "NAO INFORMADO",
    "NA",
    "N/A"
]

for coluna in colunas_ausencia:
    df_silver = df_silver.withColumn(
        coluna,
        F.when(
            F.upper(F.trim(F.col(coluna))).isin(marcadores_ausencia),
            F.lit(None)
        ).otherwise(F.col(coluna))
    )

In [0]:
display(
    df_silver.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in colunas_ausencia
    ])
)

condicao_metereologica,sentido_via,regional,delegacia,uop,classificacao_acidente
4492,883,27,127,282,5


### Resultado da padronização categórica

Foi realizada a padronização da categoria `Transitar no Acostamento`, eliminando a variação de capitalização identificada durante o profiling.

Também foram convertidos para `NULL` os valores textuais utilizados para representar ausência de informação nas colunas analisadas, como `IGNORADO`, `NÃO INFORMADO`, `NA` e `N/A`.

A transformação foi aplicada apenas às colunas em que esses marcadores haviam sido identificados durante a análise de qualidade.

Essa padronização permite distinguir valores realmente informados de registros sem informação disponível, facilitando filtros, agregações e análises posteriores.

## Criação de Atributos Derivados

Após a correção dos tipos e a padronização dos valores categóricos, serão criados novos atributos derivados dos dados existentes.

Esses campos têm como objetivo facilitar as análises posteriores sem alterar as informações originais.

Serão criados atributos relacionados ao período da ocorrência e à gravidade do acidente.

In [0]:
df_silver = (
    df_silver

    .withColumn(
        "ano",
        F.year(F.col("data_inversa"))
    )

    .withColumn(
        "mes",
        F.month(F.col("data_inversa"))
    )

    .withColumn(
        "hora",
        F.hour(F.to_timestamp(F.col("horario"), "HH:mm:ss"))
    )

    .withColumn(
        "fim_de_semana",
        F.when(
            F.col("dia_semana").isin("sábado", "domingo"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "acidente_fatal",
        F.when(
            F.col("mortos") > 0,
            1
        ).otherwise(0)
    )
)

In [0]:
display(
    df_silver.select(
        "data_inversa",
        "ano",
        "mes",
        "horario",
        "hora",
        "dia_semana",
        "fim_de_semana",
        "mortos",
        "acidente_fatal"
    ).limit(20)
)

data_inversa,ano,mes,horario,hora,dia_semana,fim_de_semana,mortos,acidente_fatal
2025-01-01,2025,1,06:20:00,6,quarta-feira,0,0,0
2025-01-01,2025,1,07:50:00,7,quarta-feira,0,1,1
2025-01-01,2025,1,08:45:00,8,quarta-feira,0,0,0
2025-01-01,2025,1,11:00:00,11,quarta-feira,0,0,0
2025-01-01,2025,1,09:30:00,9,quarta-feira,0,0,0
2025-01-01,2025,1,10:40:00,10,quarta-feira,0,2,1
2025-01-01,2025,1,12:23:00,12,quarta-feira,0,0,0
2025-01-01,2025,1,17:45:00,17,quarta-feira,0,0,0
2025-01-01,2025,1,18:40:00,18,quarta-feira,0,1,1
2025-01-01,2025,1,17:00:00,17,quarta-feira,0,0,0


In [0]:
df_silver.groupBy("fim_de_semana").count().show()

+-------------+------+
|fim_de_semana| count|
+-------------+------+
|            1|112389|
|            0|230235|
+-------------+------+



In [0]:
df_silver.groupBy("acidente_fatal").count().show()

+--------------+------+
|acidente_fatal| count|
+--------------+------+
|             1| 24619|
|             0|318005|
+--------------+------+



In [0]:
display(
    df_silver.select(
        "data_inversa",
        "ano",
        "mes",
        "horario",
        "hora",
        "dia_semana",
        "fim_de_semana",
        "mortos",
        "acidente_fatal"
    ).limit(20)
)

data_inversa,ano,mes,horario,hora,dia_semana,fim_de_semana,mortos,acidente_fatal
2025-01-01,2025,1,06:20:00,6,quarta-feira,0,0,0
2025-01-01,2025,1,07:50:00,7,quarta-feira,0,1,1
2025-01-01,2025,1,08:45:00,8,quarta-feira,0,0,0
2025-01-01,2025,1,11:00:00,11,quarta-feira,0,0,0
2025-01-01,2025,1,09:30:00,9,quarta-feira,0,0,0
2025-01-01,2025,1,10:40:00,10,quarta-feira,0,2,1
2025-01-01,2025,1,12:23:00,12,quarta-feira,0,0,0
2025-01-01,2025,1,17:45:00,17,quarta-feira,0,0,0
2025-01-01,2025,1,18:40:00,18,quarta-feira,0,1,1
2025-01-01,2025,1,17:00:00,17,quarta-feira,0,0,0


### Resultado da criação dos atributos derivados

Foram criados novos atributos para facilitar as análises temporais e de gravidade:

- `ano`: ano da ocorrência;
- `mes`: mês da ocorrência;
- `hora`: hora extraída do campo de horário;
- `fim_de_semana`: indicador binário para sábado ou domingo;
- `acidente_fatal`: indicador binário para ocorrências com pelo menos uma vítima fatal.

Esses campos foram derivados exclusivamente de informações já existentes na base e não alteram os dados originais.

Sua criação tem como objetivo simplificar consultas e agregações nas etapas analíticas posteriores.

## Tratamento da coluna `tracado_via`

Durante o profiling foi identificado que a coluna `tracado_via` pode conter múltiplas características na mesma ocorrência, separadas por ponto e vírgula.

Como a tabela principal possui granularidade de uma linha por ocorrência, o campo original será preservado para evitar alteração da granularidade dos dados.

Adicionalmente, será criada uma coluna em formato de lista contendo as características individualizadas do traçado da via.

Essa estrutura permitirá análises futuras das características individualmente sem perda do valor originalmente fornecido pela PRF.

In [0]:
df_silver = df_silver.withColumn(
    "tracado_via_lista",
    F.transform(
        F.split(F.col("tracado_via"), ";"),
        lambda x: F.trim(x)
    )
)

In [0]:
display(
    df_silver.select(
        "tracado_via",
        "tracado_via_lista"
    )
    .filter(F.col("tracado_via").contains(";"))
    .limit(20)
)

tracado_via,tracado_via_lista
Reta;Declive,"List(Reta, Declive)"
Reta;Aclive,"List(Reta, Aclive)"
Curva;Declive,"List(Curva, Declive)"
Aclive;Curva,"List(Aclive, Curva)"
Curva;Interseção de Vias,"List(Curva, Interseção de Vias)"
Reta;Declive,"List(Reta, Declive)"
Curva;Aclive,"List(Curva, Aclive)"
Declive;Reta,"List(Declive, Reta)"
Reta;Aclive,"List(Reta, Aclive)"
Ponte;Declive;Curva,"List(Ponte, Declive, Curva)"


## Validação Final da Camada Silver

Antes da persistência da camada Silver, será realizada uma validação final para verificar se as transformações aplicadas mantiveram a quantidade de registros e se os principais campos apresentam a estrutura esperada.

In [0]:
print(f"Registros Bronze: {df_bronze.count()}")
print(f"Registros Silver: {df_silver.count()}")

Registros Bronze: 342624
Registros Silver: 342624


In [0]:
print(f"Colunas Bronze: {len(df_bronze.columns)}")
print(f"Colunas Silver: {len(df_silver.columns)}")

Colunas Bronze: 32
Colunas Silver: 38


In [0]:
df_silver.printSchema()

root
 |-- id: string (nullable = true)
 |-- data_inversa: date (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: integer (nullable = true)
 |-- mortos: integer (nullable = true)
 |-- feridos_leves: integer (nullable = true)
 |-- feridos_graves: integer (nullable = true)
 |-- ilesos: integer (nullable = true)
 |-- ignorados: integer (nullable = true)
 |-- feridos: integer (nullable = 

In [0]:
print(
    "IDs distintos na Silver:",
    df_silver.select("id").distinct().count()
)

print(
    "Total de registros na Silver:",
    df_silver.count()
)

IDs distintos na Silver: 342624
Total de registros na Silver: 342624


In [0]:
display(df_silver.limit(20))

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_arquivo,arquivo_origem,ano,mes,hora,fim_de_semana,acidente_fatal,tracado_via_lista
652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,-23.48586772,-46.54075317,SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,datatran2025.csv,2025,1,6,0,0,"List(Reta, Declive)"
652519,2025-01-01,quarta-feira,07:50:00,CE,116,546.2,PENAFORTE,Pista esburacada,Colisão frontal,null,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,-7.812288,-39.08333306,SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,datatran2025.csv,2025,1,7,0,1,List(Reta)
652522,2025-01-01,quarta-feira,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,-23.182565,-50.637228,SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,datatran2025.csv,2025,1,8,0,0,"List(Reta, Aclive)"
652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,-25.36517687,-49.04223028,SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,datatran2025.csv,2025,1,11,0,0,List(Reta)
652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,-16.46801304,-43.43121303,SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,datatran2025.csv,2025,1,9,0,0,"List(Curva, Declive)"
652569,2025-01-01,quarta-feira,10:40:00,MT,70,669.0,CACERES,Transitar na contramão,Colisão frontal,Com Vítimas Fatais,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,4,2,0,0,1,2,0,5,-16.04148578,-57.25884017,SPRF-MT,DEL03-MT,UOP02-DEL03-MT,2025,datatran2025.csv,2025,1,10,0,1,List(Reta)
652573,2025-01-01,quarta-feira,12:23:00,RS,116,376.0,TAPES,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Dupla,Reta,Não,2,0,1,0,0,1,1,2,-30.739714,-51.62594,SPRF-RS,DEL02-RS,UOP02-DEL02-RS,2025,datatran2025.csv,2025,1,12,0,0,List(Reta)
652617,2025-01-01,quarta-feira,17:45:00,SC,101,207.4,SAO JOSE,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Nublado,Dupla,Reta,Sim,2,0,1,0,1,0,1,2,-27.60001226,-48.6226467,SPRF-SC,DEL01-SC,UOP01-DEL01-SC,2025,datatran2025.csv,2025,1,17,0,0,List(Reta)
652625,2025-01-01,quarta-feira,18:40:00,MG,116,708.5,MURIAE,Velocidade Incompatível,Tombamento,Com Vítimas Fatais,Anoitecer,Crescente,Nublado,Simples,Curva,Não,2,1,0,0,0,1,0,2,-21.16328873,-42.37968988,SPRF-MG,DEL07-MG,UOP02-DEL07-MG,2025,datatran2025.csv,2025,1,18,0,1,List(Curva)
652648,2025-01-01,quarta-feira,17:00:00,PE,407,7.4,AFRANIO,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Aclive;Curva,Não,2,0,0,0,2,0,0,1,-8.47503105,-41.0137105,SPRF-PE,DEL06-PE,UOP02-DEL06-PE,2025,datatran2025.csv,2025,1,17,0,0,"List(Aclive, Curva)"


### Resultado da validação final

Após a aplicação das transformações, a camada Silver manteve os 342.624 registros originalmente presentes na camada Bronze.

Não houve perda ou duplicação de ocorrências durante o processo de transformação.

Os campos foram convertidos para tipos adequados, os valores categóricos identificados como inconsistentes foram padronizados, os marcadores textuais de ausência foram convertidos para valores nulos e foram criados atributos derivados para facilitar as análises posteriores.

O campo `tracado_via` foi preservado em sua forma original e complementado pela coluna `tracado_via_lista`, permitindo representar individualmente suas características sem alterar a granularidade da tabela.

A camada Silver encontra-se, portanto, pronta para persistência e utilização nas próximas etapas de modelagem e análise.

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.prf_acidentes_silver")
)

In [0]:
spark.table(
    "workspace.default.prf_acidentes_silver"
).count()

342624